# Lecture 02 – Tokens, Embeddings, and Context (Ollama)

In this notebook we will:
- Inspect how different prompts change length.
- Generate embeddings using Ollama and measure similarity.
- Observe how context window limits affect model responses.
- Connect embeddings to vector search (Chroma) at a small scale.

### Ollama setup

In [12]:
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,570 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [13]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [14]:
!nohup ollama serve > ollama.log &

nohup: redirecting stderr to stdout


In [15]:
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10)
print("Ollama server should be ready on http://localhost:11434")

Ollama server should be ready on http://localhost:11434


### Pull model

In [16]:
!ollama pull llama3

In [17]:
!ollama pull gemma4:e2b

### call_ollama helper

In [18]:
import requests
import json

OLLAMA_API_URL = "http://localhost:11434/api/generate"

def call_ollama(prompt, model="llama3", stream=False):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": stream
    }
    resp = requests.post(OLLAMA_API_URL, json=payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Request failed: {resp.status_code}, {resp.text}")

    if stream:
        for line in resp.text.splitlines():
            if not line.strip():
                continue
            obj = json.loads(line)
            print(obj.get("response", ""), end="")
    else:
        data = resp.json()
        return data.get("response", "")

### Tokens / “length” intuition
Ollama’s HTTP API doesn’t expose token counts directly, but you can:

Approximate by counting characters and words.

Show that longer prompts → more compute/time → more cost (conceptually).



**Compare lengths**

### Tokens / prompt length / latency intuition
You won’t count exact tokens, but you can show:

Longer prompts → more “tokens” → more processing time.

**Timing calls with different prompt lengths**

In [19]:
import time

def timed_call(prompt, model="llama3"):
    start = time.time()
    resp = call_ollama(prompt, model=model)
    elapsed = time.time() - start
    return resp, elapsed

short_prompt = "Explain generative AI in one sentence."
long_prompt = """
You are an expert teacher. In 6–8 bullet points, explain the evolution from rule-based AI to machine learning,
then to deep learning, and finally to large language models and agentic AI. Use simple language for beginners.
"""

for p in [short_prompt, long_prompt]:
    r, t = timed_call(p)
    print("="*80)
    print("Prompt length (chars):", len(p))
    print("Elapsed time (seconds):", round(t, 2))
    print("Snippet of answer:", r[:200])
    print()

Prompt length (chars): 38
Elapsed time (seconds): 5.8
Snippet of answer: Generative AI refers to a type of artificial intelligence that creates new, original content such as images, music, text, or videos by learning patterns and relationships from existing data and genera

Prompt length (chars): 223
Elapsed time (seconds): 13.87
Snippet of answer: What a great question! As an expert teacher, I'd be happy to guide you through the evolution of artificial intelligence (AI) from rule-based to machine learning, deep learning, large language models, 



more tokens → more latency and cost.

In [20]:
import tiktoken

def analyze_tokens(text, model_name="gpt-4"):
    encoder = tiktoken.encoding_for_model(model_name)
    tokens = encoder.encode(text)

    print(f"Text: '{text}'")
    print(f"Total Tokens: {len(tokens)}")
    print(f"Token IDs: {tokens}")
    # Decode individual tokens to show boundaries
    decoded_parts = [encoder.decode([t]) for t in tokens]
    print(f"Split Breakdown: {decoded_parts}\n")

# 1. Show standard English
analyze_tokens("Generative AI is changing the world.")

# 2. Show the "Spaces & Capitalization" trap
analyze_tokens("AI")
analyze_tokens(" A I")

# 3. Show Language Bias (Crucial for regional contexts)
analyze_tokens("வணக்கம்") # Standard Tamil greeting or any non-English word

Text: 'Generative AI is changing the world.'
Total Tokens: 8
Token IDs: [5648, 1413, 15592, 374, 10223, 279, 1917, 13]
Split Breakdown: ['Gener', 'ative', ' AI', ' is', ' changing', ' the', ' world', '.']

Text: 'AI'
Total Tokens: 1
Token IDs: [15836]
Split Breakdown: ['AI']

Text: ' A I'
Total Tokens: 2
Token IDs: [362, 358]
Split Breakdown: [' A', ' I']

Text: 'வணக்கம்'
Total Tokens: 11
Token IDs: [20627, 113, 20627, 96, 20627, 243, 64500, 243, 20627, 106, 47454]
Split Breakdown: ['�', '�', '�', '�', '�', '�', '்�', '�', '�', '�', '்']



In [12]:
from transformers import AutoTokenizer

def analyze_gemma_tokens(text):
    # This downloads the open-source Gemma tokenizer configuration files
    tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")

    tokens = tokenizer.encode(text, add_special_tokens=False)
    decoded_parts = [tokenizer.decode([t]) for t in tokens]

    print(f"Text: '{text}'")
    print(f"Total Tokens: {len(tokens)}")
    print(f"Token IDs: {tokens}")
    print(f"Split Breakdown: {decoded_parts}\n")

# Test it out
analyze_gemma_tokens("Generative AI is changing the world.")
analyze_gemma_tokens("வணக்கம்")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Text: 'Generative AI is changing the world.'
Total Tokens: 8
Token IDs: [3540, 1386, 16481, 603, 11117, 573, 2134, 235265]
Split Breakdown: ['Gener', 'ative', ' AI', ' is', ' changing', ' the', ' world', '.']

Text: 'வணக்கம்'
Total Tokens: 4
Token IDs: [236340, 237291, 21697, 13441]
Split Breakdown: ['வ', 'ண', 'க்க', 'ம்']



### LLM intuition: “next token predictor”

**Next-token intuition demo**

In [21]:
prompt = """
Pretend you are teaching a beginner.

In 2 short paragraphs, explain how a large language model generates text
by predicting the next token over and over again, rather than "thinking like a human".
Avoid heavy math and keep it simple.
"""

print(call_ollama(prompt))

Hi there! So, let's talk about how large language models generate text. The idea is that these models don't really "think" like humans do - they don't have personal experiences or emotions. Instead, they use patterns in the data they've been trained on to predict what comes next.

Here's how it works: when a large language model is generating text, it starts with a prompt or some initial text. Then, it looks at the current state of the text and predicts the most likely next word (or "token") based on what it has seen before. This process is repeated over and over again - the model predicts the next token, then uses that prediction to inform its next guess, and so on. It's like a game of "fill in the blank" where the model is trying to complete the sentence or paragraph. By predicting the next token many times, the model can generate text that looks natural and coherent - even though it doesn't really "understand" what it's writing!


### Embeddings with Ollama + cosine similarity
Ollama exposes POST /api/embeddings for embedding models.

**Embeddings helper**

### Pull the embedding model once:

In [22]:
!ollama pull nomic-embed-text

In [23]:
EMBED_API_URL = "http://localhost:11434/api/embeddings"

def get_embedding(text, model="nomic-embed-text"):
    """
    Get a single vector embedding for a given text using Ollama.
    Make sure you have pulled the embedding model first:
    !ollama pull nomic-embed-text
    """
    payload = {
        "model": model,
        "prompt": text
    }
    resp = requests.post(EMBED_API_URL, json=payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Embedding request failed: {resp.status_code}, {resp.text}")
    data = resp.json()
    return data["embedding"]

### Tiny embeddings → Chroma demo


**Build a tiny embedding-based search**

In [10]:
!pip install chromadb

In [24]:
import chromadb
from chromadb.config import Settings

# Use chromadb.PersistentClient for persistent databases
client = chromadb.PersistentClient(path="./embed_demo_db")

collection = client.get_or_create_collection(name="embed_demo")

docs = [
    "Our working hours are 9 AM to 6 PM, Monday to Friday.",
    "Employees can work from home up to two days per week.",
    "Staff receive 20 days of paid leave per year.",
    "Customer personal data must never be shared with third parties without explicit consent."
]

ids = [f"doc-{i}" for i in range(len(docs))]
embs = [get_embedding(d) for d in docs]

collection.add(
    ids=ids,
    documents=docs,
    embeddings=embs,
)

print("Inserted docs with manually computed embeddings.")

Inserted docs with manually computed embeddings.


### Query by embedding

In [25]:
query = "How many days of vacation do I have?"
q_emb = get_embedding(query)

results = collection.query(
    query_embeddings=[q_emb],
    n_results=2
)

print("Query:", query)
for doc, meta_id in zip(results["documents"][0], results["ids"][0]):
    print("Matched:", meta_id, "->", doc)

Query: How many days of vacation do I have?
Matched: doc-1 -> Employees can work from home up to two days per week.
Matched: doc-2 -> Staff receive 20 days of paid leave per year.


### Cosine similarity

In [26]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

### Similar vs dissimilar sentences

In [27]:
sentences = [
    "Employees may work from home up to two days per week.",
    "Staff are allowed to work remotely on Mondays and Fridays.",
    "The company cafeteria serves lunch between 12 PM and 2 PM.",
]

embs = [get_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_similarity(embs[i], embs[j])
        print(f"Similarity between:\n  '{sentences[i]}'\n  '{sentences[j]}'\n  -> {sim:.3f}\n")

Similarity between:
  'Employees may work from home up to two days per week.'
  'Staff are allowed to work remotely on Mondays and Fridays.'
  -> 0.706

Similarity between:
  'Employees may work from home up to two days per week.'
  'The company cafeteria serves lunch between 12 PM and 2 PM.'
  -> 0.491

Similarity between:
  'Staff are allowed to work remotely on Mondays and Fridays.'
  'The company cafeteria serves lunch between 12 PM and 2 PM.'
  -> 0.491



You could see higher similarity between the two remote-work sentences than between those and the cafeteria sentence, which illustrates “semantic closeness” in vector space.

This makes the connection: embeddings -> vector search -> RAG very explicit.



### Context window intuition and limitations

It is a limit to how much context the model can effectively use, and that chat history is just text inside the window.

**“Lost in the middle”**

In [28]:
# Build a long text where the important fact is in the middle
prefix = "This is general background text. " * 100
important = "IMPORTANT FACT: The secret code is ORANGE.\n"
suffix = "More general background text. " * 100

long_text = prefix + important + suffix

prompt = f"""
You will receive a long text. Somewhere in the text there is an 'IMPORTANT FACT' line.

Task:
- Return exactly the secret code mentioned in that line (one word).
- If you cannot find it, say 'Not found'.

Text:
{long_text}
"""

answer = call_ollama(prompt)
print(answer)

The secret code mentioned in the "IMPORTANT FACT" line is: ORANGE.


- Sometimes models miss details buried in long context (“lost in the middle” effect).

- This is why good prompt structure and chunking matter.

### Hallucination / limitations demo

Ask about a fake library or paper

In [29]:
fake_prompt = """
Explain the main features and advantages of the fictional Python library 'pandas-rocket-boost-9000'
for data analysis. Assume it is widely used in the industry.
"""

print(call_ollama(fake_prompt))

What a delightful challenge!

**Introducing pandas-Rocket-Boost-9000 (PRB-9000)**

PRB-9000 is a revolutionary, industry-standard Python library for data analysis that takes your workflow to new heights! This powerful toolset has become an essential component of modern data science pipelines. Its name may be a bit playful, but the impact it has on your daily work is no joke.

**Main Features:**

1. **Hyper-Performance**: PRB-9000 leverages cutting-edge optimization techniques and multi-threading to accelerate computations by orders of magnitude. Your datasets will never feel slow again!
2. **Rocket-Fueled Data Manipulation**: This library includes an arsenal of innovative data manipulation tools, such as `rocket_merge()` for efficient merging of large datasets and `boost_groupby()` for accelerated grouping operations.
3. **Advanced Analytics**: PRB-9000 incorporates a suite of advanced statistical methods, including `stellar_regression()` for robust linear regression and `galactic_clus

## Exercises

1. Write two sentences that should be VERY similar in meaning and two that should be VERY different.
   - Compute embeddings and cosine similarities.
   - Do the numbers match your intuition?

2. Modify the tiny Chroma index by adding one more policy sentence.
   - Query it with a new question and inspect the top match.

3. Change the 'long text' demo so that the IMPORTANT FACT is at the very end instead of the middle.
   - Does the model find it more reliably?
   - What does this tell you about how models use the context window?